# Accuracy Analyzer Tool
The accuracy analyzer is used to debug accuracy issues with a given model on AIC with one specific input data.

The tool does the accuracy debugging by comparing raw tensor values w.r.t to some reference framework. The reference framework could be Onnx Runtime for onnx models, the AIC interpreter, Tensorflow and Caffe

## Requirements 
- Qualcomm Cloud AI 100 device
- Qualcomm AI 100 Software Development Kit >= 1.7
   - Platform SDK
   - Apps SDK 
- Python 3.8*-devel 
     - Centos: yum install python3-devel
     - Ubuntu: apt-get install python3-dev

### Setting up the Environment

#### Adding Kernel to Jupyter Notebook/Lab 


In [1]:
!/opt/qti-aic/dev/python/qaic-env/bin/python -m pip install ipykernel 
!source /opt/qti-aic/dev/python/qaic-env/bin/activate &&  /opt/qti-aic/dev/python/qaic-env/bin/python -m ipykernel install --user --name qaic-env --display-name "Python (qaic-env)";

Badly placed ()'s.




## Tool usage and options

### Comparators

This option enables user to choose what kind of comparison to use to evaluate layerwise similarity between AIC and reference outputs. One can pass multiple comparators with this option

If the match is not 100%, its marked as a mismatched layer and the model shall be partitioned at that layer when using layerwise snooping.

Valid options:

 abs | rme | avg | l1norm | l2norm | cos | kld | std 

**Description of each comparator and its formulae:**

*Note:* Here op1 and op1 are the numpy output tensors from aic and reference platform 

**abs:**  
evaluates the absolute percent mismatch between two tensors (elementwise). Uses a tolerance threshold to control the allowed % deviation.
```
match = np.allclose(op1, op2, rtol=self._tol, equal_nan=True)
tdiff = np.isclose(op1, op2, rtol=self._tol, equal_nan=True)
common = op1[tdiff]
match_percent = (common.shape[0] * 100) / op2.shape[0]
```

**rme:**
Evaluates the RMSE between the two tensors.
```
rmse = np.linalg.norm(op2 - op1) / np.sqrt(len(op1))
match_percent = 100 - rmse
 ```       

**avg:**
Evaluates the error between the two tensors by dividing the absolute gap by average value of the tensors.
```
avg_value = np.nan_to_num(max(np.average(np.absolute(op1)), np.average(np.absolute(op2))))
if (avg_value != 0):
    avg_error = np.average(abs(op1 - op2)) / avg_value
else:
    avg_error = 0.0
avg_error *= 100.0
match_percent = 100 - avg_error
```

**l1norm, l2norm:**
Evaluates the L1 and L2 norm of the two tensors.
```
ord = 1 for l1norm
ord = 2 for l2nor  
diff = np.linalg.norm(op2 - op1, ord=1)
ref_norm = np.linalg.norm(op1, ord=1)
match_percent = 100 - (diff * 100 / ref_norm)
```

**cos:**
Evaluates the cosine similarity between the two tensors
```
cos_sim = np.dot(op1, op2)/ (np.linalg.norm(op1) * np.linalg.norm(op2))
match_percent = (1+cos_sim)/2 * 100
```

**kld:**
Evaluates the KL Divergence between the two tensors
```
kld = entropy(op1, op2, base=2)
```

**std:**
Evaluates the Relative Standard deviation between the two tensors Comparator is enabled by default and the tool supports various set of in built comparators.By default, average comparator is used.
```
std1 = np.std(op1)
std2 = np.std(op2)
pct_change = (abs(std2 - std1) / std1)*100
```


In [16]:
# Get the defaults from SDK into current work directory if doesnt exist.
import os
import shutil
import pandas as pd
import sys
if not os.path.exists('defaults.yaml'):
    !cp /opt/qti-aic/tools/qaic-pytools/defaults.yaml .
    

cwd = os.getcwd()
# delete old log directories
if os.path.exists(os.path.join(cwd , 'log_dir')):
    shutil.rmtree(os.path.join(cwd , 'log_dir'))
    
device_id = '0'
model_path = 'Resnet50/resnet50_v1.onnx'
list_file = 'Resnet50/list.txt'
input_file = 'Resnet50/dog.raw'
os.system('echo \"dog.raw\" > Resnet50/list.txt')

pgq_log = os.path.join(cwd ,'log_dir/pgq_log_dir')


# PGQ Sweep
This strategy is useful to identify the best possible PGQ combination. In this strategy, we do Oneshot snooping over all the possible PGQ combinations and identify the best possible combination. This is a good starting point to use the accuracy analyzer. After identifying the best possible combination user can try our various other strategies and snooping methods to identifying the sensitive nodes/sub-graphs. User can reduce the combinations to search by giving custom combinations with pgq-options parameter.

**Note:** This step is not required if user has results of Accuracy Evaluator

## Executing pgq sweep:

In [17]:
!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
        -model {model_path} \
        -input-list {list_file} \
        -work-dir {pgq_log} \
        -reference onnxrt \
        -precision int8 \
        -strategy pgq-sweep 

/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/paramiko/transport.py:236: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
INFO: QAIC Model Analyzer
INFO: Cleaned and Optimized model is saved at /local/mnt/workspace/github/resnet/Analyzer/log_dir/pgq_log_dir/cleanmodel.onnx
INFO: Total no of nodes in the cleaned graph : 125
INFO: Node types and count in the cleaned graph:
   op_type  count
      Conv     53
      Relu     49
       Add     17
   MaxPool      1
ReduceMean      1
   Squeeze      1
    MatMul      1
   Softmax      1
    ArgMax      1
INFO: Snooping all intermediate outputs of model
INFO: Captured reference onnxrt session data
INFO: Captured onnx profile
INFO: Compiling and executing original model...
                  Layer Type      Shape Data Type Ref Activations (min,max,median) Aic Activations (Min,Max,Median)    Avg(%) Info
O/P Name                                                                              

### PGQ sweep result:
Best PGQ profile : quantization-schema:symmetric_with_uint8, quantization-calibration: None, channelwise: True, percentile-calibration-value: None

In [65]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
csv_path = os.path.join(cwd ,'log_dir/pgq_log_dir/pgq_sweep.csv')
df=pd.read_csv(csv_path)
df
# extra_options = '-aic-compile-args="quantization-schema=symmetric_with_uint8;enable-channelwise"'

,Index,quant-schema,quant-calib,channelwise,% calibration,mean match%,Orig Outputs
0,1,asymmetric,None,True,-,85.2417,"{'473': OrderedDict([('Avg(%)', 85.2417)])}"
1,2,asymmetric,None,False,-,79.5983,"{'473': OrderedDict([('Avg(%)', 79.5983)])}"
2,3,asymmetric,KLMinimization,True,-,83.7010,"{'473': OrderedDict([('Avg(%)', 83.701)])}"
3,4,asymmetric,KLMinimization,False,-,78.3025,"{'473': OrderedDict([('Avg(%)', 78.3025)])}"
4,5,asymmetric,Percentile,True,99.9,75.6176,"{'473': OrderedDict([('Avg(%)', 75.6176)])}"
5,6,asymmetric,Percentile,True,99.99,85.4364,"{'473': OrderedDict([('Avg(%)', 85.4364)])}"
6,7,asymmetric,Percentile,True,99.999,85.5984,"{'473': OrderedDict([('Avg(%)', 85.5984)])}"
7,8,asymmetric,Percentile,True,99.9999,85.6586,"{'473': OrderedDict([('Avg(%)', 85.6586)])}"
8,9,asymmetric,Percentile,False,99.9,72.7009,"{'473': OrderedDict([('Avg(%)', 72.7009)])}"
9,10,asymmetric,Percentile,False,99.99,79.1987,"{'473': OrderedDict([('Avg(%)', 79.1987)])}"


# Oneshot Snooping
In this snooping mode, on a single run user can configure the tool to dump all intermediate layer activations of the model on AIC and compare them with golden outputs using one or more in built or custom comparators.

![flow](images/oneshot_flow.png)

Below image gives a view of sample model transformation in oneshot snooping
![flow](images/oneshot_model_tranforms.png)

### Executing the oneshot run of Accuracy Analyzer tool



In [9]:

!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
           -reference onnxrt \
           -snooping oneshot \
           -comparator cos \
           -tol-thres 0.01 \
           -model {model_path} \
           -input-list {list_file} \
           -work-dir 'log_dir/oneshot_log_dir' \
           -skip-initial-run \
           -precision int8 \
           -device-id {device_id} \ 
           -silent 

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/home/adarisi/ml-tools/qaic-pytools-dist/env/lib64/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} enabled. "

### Oneshot result:

In [67]:

pd.set_option("display.max_rows", None, "display.max_columns", None)
csv_path = os.path.join(cwd,'log_dir/oneshot_log_dir/oneshot.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Layer Type,Shape,"Activations (min,max,median)",CosineSimilarity(%),Info
0,473,(*)Gemm,"(1, 1000)","(-7.962, 14.355, -0.18)",97.9618,-
1,317,Clip,"(1, 32, 112, 112)","(0.0, 2.962, 0.244)",99.9911,-
2,320,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",99.9216,-
3,322,Conv,"(1, 16, 112, 112)","(-6.109, 5.047, -0.001)",99.0808,-
4,325,Clip,"(1, 96, 112, 112)","(0.0, 4.287, 0.154)",99.6197,-
5,328,Clip,"(1, 96, 56, 56)","(0.0, 3.188, 0.098)",99.5639,-
6,330,Conv,"(1, 24, 56, 56)","(-2.589, 3.013, -0.01)",98.0537,-
7,333,Clip,"(1, 144, 56, 56)","(0.0, 1.068, 0.13)",99.2260,-
8,336,Clip,"(1, 144, 56, 56)","(0.0, 2.844, 0.047)",99.3472,-
9,338,Conv,"(1, 24, 56, 56)","(-3.349, 3.994, -0.009)",98.5688,-


# Layerwise Snooping
In Layerwise debugging, the debugger debugs one layer at a time.  The reference activations for all the layers are collected first by running the model on a reference platform.

Next, layer outputs are enabled one a time, the model is compiled and executed on AIC in an iterative manner. Just like Oneshot snooping, currently the debugger modifies the original model to enable the intermediate output nodes.

The AIC outputs are compared with the reference for the added layer and the original outputs - in each iteration (using one or more comparators).

If the comparison resulted in a mismatch with reference, the debugger partitions the model after that layer and feeds the reference output for that layer to the new model and the debugging process continues with the new model.

Since there is compilation (additionally profile generation when run in quantized int8 precision) followed by execution at each layer, the debugging process is time consuming. There are multiple options which can be used to speed up this debugging process like starting to debug from a specific layer, skipping layer types, top-partition mode, etc.. as described in the debugging options section.

![flow](images/layerwise_flow.png)

Below image gives a view of sample model transformations in each iteration of layerwise snooping

![flow](images/layerwise_model_transforms.PNG)

## Executing the layerwise run of Accuracy Analyzer tool



In [91]:

!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                -reference onnxrt \
                -snooping layerwise \
                -comparator cos \
                -tol-thres 0.01 \
                -model {model_path} \
                -input-list {list_file} \
                -work-dir 'log_dir/layerwise_log_dir' \
                -skip-initial-run \
                -precision int8 \
                -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Layerwise result:

In [92]:
csv_path = os.path.join(cwd , 'log_dir/layerwise_log_dir/layerwise.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Status,Layer Type,Shape,"Activations (Min,Max,Median)",CosineSimilarity(%),Orig Outputs,Info
0,317,part,Clip,"(1, 32, 112, 112)","(0.0, 2.962, 0.244)",99.9911,{'473': {'CosineSimilarity(%)': 97.9618}},-
1,320,part,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",99.9290,{'473': {'CosineSimilarity(%)': 98.024}},-
2,322,part,Conv,"(1, 16, 112, 112)","(-6.109, 5.047, -0.001)",99.9467,{'473': {'CosineSimilarity(%)': 99.2711}},-
3,325,part,Clip,"(1, 96, 112, 112)","(0.0, 4.287, 0.154)",99.9758,{'473': {'CosineSimilarity(%)': 99.2837}},-
4,328,part,Clip,"(1, 96, 56, 56)","(0.0, 3.188, 0.098)",99.9633,{'473': {'CosineSimilarity(%)': 99.268}},-
5,330,part,Conv,"(1, 24, 56, 56)","(-2.589, 3.013, -0.01)",99.9802,{'473': {'CosineSimilarity(%)': 99.5101}},-
6,333,part,Clip,"(1, 144, 56, 56)","(0.0, 1.068, 0.13)",99.9901,{'473': {'CosineSimilarity(%)': 99.5233}},-
7,336,part,Clip,"(1, 144, 56, 56)","(0.0, 2.844, 0.047)",99.9798,{'473': {'CosineSimilarity(%)': 99.4945}},-
8,338,part,Conv,"(1, 24, 56, 56)","(-3.349, 3.994, -0.009)",99.9787,{'473': {'CosineSimilarity(%)': 99.546}},-
9,339,part,Add,"(1, 24, 56, 56)","(-4.153, 5.757, -0.012)",99.9845,{'473': {'CosineSimilarity(%)': 99.5535}},-


### Graphical result of layerwise run
Below is the plot between nodes vs cosine similarity value between aic model output and onnxrt model output  
![plot](images/demo_plot_2.png)

## Executing layerwise run to extract sub-graph
   This run extracts the subgraph from model at specified start and end layers

In [68]:

!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                -reference onnxrt \
                -snooping layerwise \
                -comparator cos \
                -tol-thres 0.01 \
                -model {model_path} \
                -input-list {list_file} \
                -work-dir 'log_dir/layerwise_extract_log_dir' \
                -skip-initial-run \
                -precision int8 \
                -start-from-layer-output 371 \
                -end-layer-output 379 \
                -extract \
                -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

## Executing layerwise run on sub-graph 
 This run gives the layerwise snooping results on specified sub-graph of model

In [70]:

!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                -reference onnxrt \
                -snooping layerwise \
                -comparator cos \
                -tol-thres 0.01 \
                -model {model_path} \
                -input-list {list_file} \
                -work-dir 'log_dir/layerwise_subgraph_log_dir' \
                -skip-initial-run \
                -precision int8 \
                -start-from-layer-output 371 \
                -end-layer-output 379 \
                -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Layerwise run result

In [74]:
csv_path = os.path.join(cwd, 'log_dir/layerwise_subgraph_log_dir/layerwise.csv')
df=pd.read_csv(csv_path)
df.fillna('', inplace=True)
df

,O/P Name,Status,Layer Type,Shape,"Activations (Min,Max,Median)",CosineSimilarity(%),Orig Outputs,Info
0,373,part,Conv,"(1, 64, 14, 14)","(-1.992, 2.155, -0.019)",99.9857,{'379': {'CosineSimilarity(%)': 99.9562}},-
1,376,part,Clip,"(1, 384, 14, 14)","(0.0, 0.875, 0.102)",99.9879,{'379': {'CosineSimilarity(%)': 99.9605}},-
2,379,,(*)Clip,"(1, 384, 14, 14)","(0.0, 1.127, 0.0)",99.9667,{'379': {'CosineSimilarity(%)': 99.9667}},-


## Executing Layerwise snooping on specific op type
This run gives the layerwise snooping only on specific op type (Add) of model

*Note: User can use **-add-layer-outputs** option to specify particular node to be debugged* 

In [75]:

!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                 -reference onnxrt \
                 -snooping layerwise \
                 -comparator cos \
                 -tol-thres 0.01 \
                 -model {model_path} \
                 -input-list {list_file} \
                 -work-dir 'log_dir/layerwise_add_layer_log' \
                 -skip-initial-run \
                 -precision int8 \
                 -add-layers Add \
                 -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Layerwise run result

In [76]:
csv_path = os.path.join(cwd , 'log_dir/layerwise_add_layer_log/layerwise.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Status,Layer Type,Shape,"Activations (Min,Max,Median)",CosineSimilarity(%),Orig Outputs,Info
0,317,skip,Clip,"(1, 32, 112, 112)","(0.0, 2.962, 0.244)",-,-,-
1,320,skip,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",-,-,-
2,322,skip,Conv,"(1, 16, 112, 112)","(-6.109, 5.047, -0.001)",-,-,-
3,325,skip,Clip,"(1, 96, 112, 112)","(0.0, 4.287, 0.154)",-,-,-
4,328,skip,Clip,"(1, 96, 56, 56)","(0.0, 3.188, 0.098)",-,-,-
5,330,skip,Conv,"(1, 24, 56, 56)","(-2.589, 3.013, -0.01)",-,-,-
6,333,skip,Clip,"(1, 144, 56, 56)","(0.0, 1.068, 0.13)",-,-,-
7,336,skip,Clip,"(1, 144, 56, 56)","(0.0, 2.844, 0.047)",-,-,-
8,338,skip,Conv,"(1, 24, 56, 56)","(-3.349, 3.994, -0.009)",-,-,-
9,339,part,Add,"(1, 24, 56, 56)","(-4.153, 5.757, -0.012)",98.4206,{'473': {'CosineSimilarity(%)': 97.9618}},-


## Executing layerwise run with skip layers
This run gives the layerwise snooping results on model by skipping specified op types (Conv,Clip,Add

*Note: User can use **-skip-layer-outputs** option to specify particular node to be skipped* 

In [77]:
!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                 -reference onnxrt \
                 -snooping layerwise \
                 -comparator cos \
                 -tol-thres 0.01 \
                 -model {model_path} \
                 -input-list {list_file} \
                 -work-dir 'log_dir/layerwise_skip_layer_log' \
                 -skip-initial-run \
                 -precision int8 \
                 -skip-layers Conv,Clip,Add \
                 -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Layerwise run result

In [78]:
csv_path = os.path.join(cwd , 'log_dir/layerwise_skip_layer_log/layerwise.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Status,Layer Type,Shape,"Activations (Min,Max,Median)",CosineSimilarity(%),Orig Outputs,Info
0,317,skip,Clip,"(1, 32, 112, 112)","(0.0, 2.962, 0.244)",-,-,-
1,320,skip,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",-,-,-
2,322,skip,Conv,"(1, 16, 112, 112)","(-6.109, 5.047, -0.001)",-,-,-
3,325,skip,Clip,"(1, 96, 112, 112)","(0.0, 4.287, 0.154)",-,-,-
4,328,skip,Clip,"(1, 96, 56, 56)","(0.0, 3.188, 0.098)",-,-,-
5,330,skip,Conv,"(1, 24, 56, 56)","(-2.589, 3.013, -0.01)",-,-,-
6,333,skip,Clip,"(1, 144, 56, 56)","(0.0, 1.068, 0.13)",-,-,-
7,336,skip,Clip,"(1, 144, 56, 56)","(0.0, 2.844, 0.047)",-,-,-
8,338,skip,Conv,"(1, 24, 56, 56)","(-3.349, 3.994, -0.009)",-,-,-
9,339,skip,Add,"(1, 24, 56, 56)","(-4.153, 5.757, -0.012)",-,-,-


### Executing Layerwise run on model with skip_layer_patterns option
This run performs layerwise run on all nodes of model by skipping node fusion patterns provided by user.


In this case, user needs to provide yaml file which contains the list of layer patterns to be skipped

eg : [['Conv','Relu'],['Conv','Batchnorm','Relu']]

In [60]:
!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                -reference onnxrt \
                -snooping layerwise \
                -comparator cos \
                -tol-thres 0.01 \
                -model {model_path} \
                -input-list {list_file} \
                -work-dir 'log_dir/layerwise_skiplayer_pattern_log_dir' \
                -skip-initial-run \
                -precision int8 \
                -skip-layer-patterns 'mobilenet/patterns.yaml' \
                -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Layerwise run result

In [3]:
csv_path = os.path.join(cwd , 'sample_log_dir/log_dir/layerwise_skiplayer_pattern_log_dir/layerwise.csv')
df=pd.read_csv(csv_path)
df

FileNotFoundError: [Errno 2] No such file or directory: '/home/adarisi/Analyzer_demo/sample_log_dir/log_dir/layerwise_skiplayer_pattern_log_dir/layerwise.csv'

## Executing Layerwise run on model with custom operator

This run preforms layerwise run on model with custom operator (QNMS) .User needs to supply custom op lib files for onnxrt and aic
as shown below.

In [79]:

!{sys.executable}  /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                    -reference onnxrt \
                    -snooping layerwise \
                    -comparator avg \
                    -tol-thres 0.01 \
                    -start-from-layer-output 728 \
                    -model 'yolo_custom_op/yolov5s_640_640_with_abp_qnms.onnx' \
                    -input-list 'yolo_custom_op/list.txt' \
                    -work-dir 'log_dir/layerwise_cus_op_log_dir' \
                    -skip-initial-run \
                    -precision fp16 \
                    -disable-check-model \
                    -disable-graph-optimization \
                    -onnx-custom-op-lib 'yolo_custom_op/libCustomQnmsYoloOrt.so' \
                    -aic-compile-args="register-custom-op=yolo_custom_op/CustomQnmsConfig.yaml" \
                    -device-id {device_id}

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
INFO: Total no of nodes in the cleaned graph : 279
INFO: Node types and count in the cleaned graph:
       op_type  count
           Mul     72
          Conv     62
       Sigmoid     62
         Slice     23
        Concat     20
           Add     11
       Reshape      9
           Sub      4
       MaxPool      3
     Transpose      3
           Pow      3
        Resize      2
           Div      2
         Split      1
     Unsqueeze      1
CustomQnmsYolo      1
INFO: Snooping all intermediate outputs of model
DEBUG: OnnxRT quantization params : None
INFO: Captured reference onnxrt session data
INFO: Captured onnx prof

### Layerwise run result 

In [80]:
csv_path = os.path.join(cwd , 'log_dir/layerwise_cus_op_log_dir/layerwise.csv')
df=pd.read_csv(csv_path)
df = df.fillna('')
df

,O/P Name,Status,Layer Type,Shape,"Activations (Min,Max,Median)",Avg(%),Orig Outputs,Info
0,733,part,Slice,"(1, 25200, 1)","(-45.332, 633.376, 274.603)",99.9818,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
1,738,part,Slice,"(1, 25200, 1)","(-36.539, 632.138, 297.882)",99.9821,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
2,743,part,Slice,"(1, 25200, 1)","(7.12, 715.263, 366.562)",99.9816,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
3,748,part,Slice,"(1, 25200, 1)","(8.154, 700.176, 339.181)",99.9818,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
4,749,part,Concat,"(1, 25200, 4)","(-45.332, 715.263, 320.191)",99.9818,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
5,750,part,Unsqueeze,"(1, 25200, 1, 4)","(-45.332, 715.263, 320.191)",99.9818,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-
6,detection_boxes,,(*)CustomQnmsYolo,"(1, 2080, 4)","(-0.117, 666.81, 0.0)",96.2250,"{'num_detections': {'Avg(%)': 99.2126}, 'detec...",-


# Strategies
Strategy mode combines various snooping techniques and identifies sensitive nodes/sub-graphs. This is easier to use and faster to run. Using strategy on a model can be helpful when you have no idea on what the sensitive parts of the graph are. There are various strategy modes that can be used to get info on critical parts of the model.

## Binary search
The goal of this strategy is to identify the top few sensitive nodes in the model. In the beginning, we do the reference run and collect the golden tensors for the outputs. Then we do OneShot Snooping, starting at various nodes in the model beginning at the input. We traverse through the model in a binary search fashion to identify the node where we see the increase in accuracy match. This will help us to find the sensitive node which might be causing the drop in the accuracy match.

![flow](images/Binary_search.PNG)

With reference to the below diagram, we initially start with a whole model as search space for the sensitive node and depending on the accuracy match at various nodes, we narrow our search space where we see higher drop in the accuracy match. At any point of time start_layer and end_layer denotes the search space to find the sensitive node. start_match and end_match denotes final graph output accuracy match when we start the Oneshot run from start_layer and end_layer respectively.

User can also use num-sensitive-nodes param to identify more than one sensitive node. But it is recommended to use Layerwise snooping or Sequential search, if the user wants to find more than couple of sensitive nodes.

*Note:* This is useful to identify 1 or 2 most sensitive nodes. This strategy is very quick to run (<15 min for 1or 2 sensitive nodes) and gives us an quick idea on what the sensitive nodes. User can then explore further in that portion of the graph to identify the issue.

### Executing Binary search strategy

In [87]:
!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                    -model {model_path} \
                    -input-list {list_file} \
                    -work-dir 'log_dir/bs_log_dir' \
                    -reference onnxrt \
                    -precision int8 \
                    -strategy binary \
                    -num-sensitive-nodes 1

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Binary search result

In [88]:
csv_path = os.path.join(cwd , 'log_dir/bs_log_dir/binary_search.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Layer Type,Shape,"Activations (Min,Max,Median)",mean match%,Orig Outputs
0,320,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",69.9453,"{'473': OrderedDict([('Avg(%)', 69.9453)])}"
1,322,Conv,"(1, 16, 112, 112)","(-6.109, 5.047, -0.001)",81.8692,"{'473': OrderedDict([('Avg(%)', 81.8692)])}"


# Sequential Search
This strategy is used to identify a sensitive sub graph as opposed to finding sensitive nodes with binary search. In this strategy we do Oneshot snooping starting with nodes at various positions (10%, 20%.......90%) and observe the final output accuracy match. This will give us information on which sub-graph has the most decrease in the output accuracy match.

We then do a Layerwise snooping on that sub-graph which has most decrease in the output accuracy match, to identify one or more sensitive nodes in that sub-graph. User can use granularity param (0-100) to change the size of the sensitive sub-graph to be identified. It is recommended to use lower granularity(5-10) for bigger models and higher granularity for smaller models (10-20).

![flow](images/Sequential_search.PNG)

*Note:* This can be used if we suspect a particular portion of the sub-graph might be sensitive. After the sequential-search, we will have an overall idea on which portions of the graph might be sensitive and will also be able to identify the sensitive nodes in that particular sub-graph.  This takes longer than binary search but way lesser time than complete model layerwise snooping, since we only do layerwise on a sensitive sub-graph

### Executing Sequential search strategy

In [89]:
!{sys.executable} /opt/qti-aic/tools/qaic-pytools/qaic-acc-analyzer.py \
                    -model {model_path} \
                    -input-list {list_file} \
                    -work-dir 'log_dir/ss_log_dir' \
                    -reference onnxrt \
                    -precision int8 \
                    -strategy sequential \
                    -granularity 10 \
                    -silent

DEBUG: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
DEBUG: Creating converter from 7 to 5
DEBUG: Creating converter from 5 to 7
INFO: Caffe engine is unsupported in this environment.
INFO: QAIC Model Analyzer
/local/mnt/workspace/DVA/ml-tools/qaic-pytools-dist/env/lib/python3.8/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:350: UserWarning: Deprecation warning. This ORT build has ['CUDAExecutionProvider', 'CPUExecutionProvider'] enabled. The next release (ORT 1.10) will require explicitly setting the providers parameter (as opposed to the current behavior of providers getting set/registered by default based on the build flags) when instantiating InferenceSession.For example, onnxruntime.InferenceSession(..., providers=["CUDAExecutionProvider"], ...)
  warnings.warn("Deprecation warning. This ORT build has {} e

### Sequential Search result

In [90]:
csv_path = os.path.join(cwd , 'log_dir/ss_log_dir/sequential_search.csv')
df=pd.read_csv(csv_path)
df

,O/P Name,Node Index,Layer Type,Shape,"Activations (Min,Max,Median)",mean match%,Orig Outputs
0,320,1,Clip,"(1, 32, 112, 112)","(0.0, 6.0, 0.321)",69.9453,"{'473': OrderedDict([('Avg(%)', 69.9453)])}"
1,336,7,Clip,"(1, 144, 56, 56)","(0.0, 2.844, 0.047)",84.8165,"{'473': OrderedDict([('Avg(%)', 84.8165)])}"
2,350,13,Clip,"(1, 192, 28, 28)","(0.0, 1.007, 0.135)",89.4926,"{'473': OrderedDict([('Avg(%)', 89.4926)])}"
3,365,20,Add,"(1, 32, 28, 28)","(-4.036, 3.924, -0.003)",88.4463,"{'473': OrderedDict([('Avg(%)', 88.4463)])}"
4,381,26,Conv,"(1, 64, 14, 14)","(-1.842, 1.862, 0.011)",88.5143,"{'473': OrderedDict([('Avg(%)', 88.5143)])}"
5,397,33,Clip,"(1, 384, 14, 14)","(0.0, 1.319, 0.0)",89.8720,"{'473': OrderedDict([('Avg(%)', 89.872)])}"
6,411,39,Clip,"(1, 576, 14, 14)","(0.0, 1.124, 0.01)",91.2683,"{'473': OrderedDict([('Avg(%)', 91.2683)])}"
7,425,45,Conv,"(1, 96, 14, 14)","(-2.265, 2.364, 0.006)",91.3650,"{'473': OrderedDict([('Avg(%)', 91.365)])}"
8,442,52,Conv,"(1, 160, 7, 7)","(-1.003, 0.807, 0.015)",93.2132,"{'473': OrderedDict([('Avg(%)', 93.2132)])}"
9,455,58,Clip,"(1, 960, 7, 7)","(0.0, 0.882, 0.0)",95.6660,"{'473': OrderedDict([('Avg(%)', 95.666)])}"


# Limitations
- Oneshot run will fail for big models like bert due to memory constraints of device
- Tool requires a onnxrt custom op lib file for models with custom-op